# Init

* Mount drive

* Install dependencies

* Navigate to necessary directory

* Read raw data

In [ ]:
%%time

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CPU times: user 12.6 ms, sys: 681 µs, total: 13.2 ms
Wall time: 840 ms


In [ ]:
%%time

!pip install sqlalchemy
!pip install tqdm
!pip install sentence_transformers
!pip install catboost
!pip install -U "sentence-transformers[train]"

CPU times: user 235 ms, sys: 37.4 ms, total: 272 ms
Wall time: 31.3 s


In [ ]:
%%time

import os
os.chdir('drive/MyDrive')
# check if directory YT exists
if not os.path.exists('YT'):
  # create directory YT
  os.makedirs('YT')
# change directory to YT
os.chdir('YT')
if not os.path.exists('data'):
  # create directory data
  os.makedirs('data')
if not os.path.exists('models'):
  # create directory models
  os.makedirs('models')
os.chdir('models')
if not os.path.exists('embeddings'):
  # create directory embeddings
  os.makedirs('embeddings')
if not os.path.exists('estimators'):
  # create directory estimators
  os.makedirs('estimators')
os.chdir('..')

CPU times: user 1.32 ms, sys: 46 µs, total: 1.36 ms
Wall time: 3.61 ms


In [ ]:
# Load data

%%time

from sqlalchemy import create_engine
import pandas as pd

db_name = 'need_for_speed_30D'

# Create a sqlite engine instance
engine = create_engine(f"sqlite:///data/{db_name}.db")

# Read the database as a dataframe
raw_df = pd.read_sql_table('video_statistics', engine)

CPU times: user 15.9 s, sys: 4.4 s, total: 20.3 s
Wall time: 44 s


# Full MiniLM (Custom)

* MinMax Scale by channel

* Use full dataset

* Create custom embeddings from MiniLM

In [ ]:
# Preprocess data

%%time

from tqdm.notebook import tqdm
tqdm.pandas

from sklearn.preprocessing import MinMaxScaler

group_data = []
for name, group in tqdm(raw_df.groupby('channel_id')):
  if len(group) > 1:
    scaler = MinMaxScaler()
    row_data = group['video_view_count'].values.reshape(-1, 1)
    group['video_view_count'] = scaler.fit_transform(row_data)
    group_data.append(group)

  0%|          | 0/1652 [00:00<?, ?it/s]

CPU times: user 4.36 s, sys: 266 ms, total: 4.62 s
Wall time: 7.33 s


In [ ]:
# Split data

%%time

df = pd.concat(group_data).dropna(subset=['video_view_count'])

my_channel_title = 'Mitchell Diedrich'
my_df = df[df['channel_title'] == my_channel_title]
df = df[df['channel_title'] != my_channel_title]

if (captioned_only := False):
  df = df[df['video_caption']==True]
if (sample := False):
  df = df.sample(frac=0.1)

from sklearn.model_selection import train_test_split

train_df, val_test_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, random_state=42)

train_df.shape, val_df.shape, test_df.shape, my_df.shape

CPU times: user 6.32 s, sys: 624 ms, total: 6.94 s
Wall time: 11.5 s


((920410, 14), (115051, 14), (115052, 14), (15, 14))

In [ ]:
%%time

# import pandas as pd
# from sentence_transformers import SentenceTransformer, InputExample, losses
# from torch.utils.data import DataLoader

# titles = train_df['video_title'].to_numpy()
# views = train_df['video_view_count'].to_numpy()

# views.shape

import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn

# Assuming you have your data as a DataFrame
titles = train_df['video_title'].to_numpy()
views = train_df['video_view_count'].to_numpy()

# Prepare data
data = pd.DataFrame({
    'title': titles,
    'views': views
})

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# use roberta
# model_name = 'all-roberta-large-v1'
model_name = 'all-MiniLM-L6-v2'

# Load SentenceTransformer model
base_model = SentenceTransformer(model_name)

# Define a regression model
class RegressionModel(nn.Module):
    def __init__(self, transformer_model):
        super(RegressionModel, self).__init__()
        self.transformer = transformer_model
        try:
          self.regressor = nn.Linear(384, 1)  # Change embedding size to 384
        except:
          self.regressor = nn.Linear(1024, 1)  # Change embedding size to 1024

    def forward(self, texts):
        # SentenceTransformer requires tokenized inputs
        embeddings = self.transformer.encode(texts, convert_to_tensor=True, show_progress_bar=True).to(device)
        # Pass the embeddings through the regression layer to predict views
        output = self.regressor(embeddings)
        return output

# Instantiate the regression model
model = RegressionModel(base_model).to(device)

if (load := True):
  model.load_state_dict(torch.load(f'models/embeddings/{model_name}_{db_name}_custom_model.pt'))
else:
  # Define loss function and optimizer
  criterion = nn.MSELoss().to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

  # Convert titles to embeddings and views to tensors
  train_titles = [row['title'] for index, row in data.iterrows()]
  train_views = torch.tensor(data['views'].values, dtype=torch.float32).unsqueeze(1).to(device)  # Reshape to (batch_size, 1)

  # Train the model
  model.train()
  for epoch in range(1):
      optimizer.zero_grad()

      # Forward pass
      predictions = model(train_titles)

      # Compute loss
      loss = criterion(predictions, train_views)

      # Backward pass and optimization
      loss.backward()
      optimizer.step()

      print(f"Epoch {epoch+1}, Loss: {loss.item()}")

  # save model as embedding model

  torch.save(model.state_dict(), f'models/embeddings/{model_name}_{db_name}_custom_model.pt')

  print("Training complete.")

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


CPU times: user 7.9 s, sys: 1.78 s, total: 9.68 s
Wall time: 20.4 s


In [ ]:
%%time

import numpy as np
from sentence_transformers import SentenceTransformer

if (load := True):
  train_embeddings = np.load(f'models/embeddings/train_embeddings_custom.npy',
                           allow_pickle=True)
  val_embeddings = np.load(f'models/embeddings/val_embeddings_custom.npy',
                          allow_pickle=True)
  test_embeddings = np.load(f'models/embeddings/test_embeddings_custom.npy',
                            allow_pickle=True)
  my_embeddings = np.load(f'models/embeddings/my_embeddings_custom.npy',
                          allow_pickle=True)
else:
  size = 4
  # Extract embeddings (using only the transformer part of your model)
  with torch.no_grad():
    train_embeddings = model.transformer.encode(
        titles,
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=size
        ).to(device)
    val_embeddings = model.transformer.encode(
        val_df['video_title'].to_numpy(),
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=size
        ).to(device)
    test_embeddings = model.transformer.encode(
        test_df['video_title'].to_numpy(),
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=size
        ).to(device)
    my_embeddings = model.transformer.encode(
        my_df['video_title'].to_numpy(),
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=size
        ).to(device)

  # convert train_embeddings to np array
  train_embeddings = train_embeddings.cpu().numpy()
  val_embeddings = val_embeddings.cpu().numpy()
  test_embeddings = test_embeddings.cpu().numpy()
  my_embeddings = my_embeddings.cpu().numpy()

  import numpy as np
  # write embeddings to files
  np.save(f'models/embeddings/train_embeddings_custom.npy', train_embeddings)
  np.save(f'models/embeddings/val_embeddings_custom.npy', val_embeddings)
  np.save(f'models/embeddings/test_embeddings_custom.npy', test_embeddings)
  np.save(f'models/embeddings/my_embeddings_custom.npy', my_embeddings)

train_embeddings.shape, val_embeddings.shape, test_embeddings.shape, my_embeddings.shape

CPU times: user 1.8 ms, sys: 3.94 s, total: 3.94 s
Wall time: 11.3 s


((920410, 384), (115051, 384), (115052, 384), (15, 384))

In [ ]:
# Fit model

%%time

pd.set_option('display.width', 1000)

from datetime import datetime as dt
from catboost import CatBoostRegressor, Pool
import time

y_train = train_df['video_view_count'].to_numpy()
y_val = val_df['video_view_count'].to_numpy()
y_test = test_df['video_view_count'].to_numpy()
y_my = my_df['video_view_count'].to_numpy()

train_pool = Pool(train_embeddings, y_train)
val_pool = Pool(val_embeddings, y_val)
test_pool = Pool(test_embeddings, y_test)
my_pool = Pool(my_embeddings, y_my)

start = time.time()

params = {
    'iterations': 8192,
    'depth': 8,
    'learning_rate': .1,
    'loss_function': 'RMSE',
    # 'l2_leaf_reg': 1000,
}

model = CatBoostRegressor(**params, random_seed=42, verbose=True, task_type='GPU')
model.fit(train_pool, eval_set=val_pool, verbose=True)


Streaming output truncated to the last 5000 lines.
3197:	learn: 0.0533865	test: 0.0633209	best: 0.0633209 (3197)	total: 2m 40s	remaining: 4m 10s
3198:	learn: 0.0533846	test: 0.0633199	best: 0.0633199 (3198)	total: 2m 40s	remaining: 4m 10s
3199:	learn: 0.0533821	test: 0.0633196	best: 0.0633196 (3199)	total: 2m 40s	remaining: 4m 10s
3200:	learn: 0.0533804	test: 0.0633195	best: 0.0633195 (3200)	total: 2m 40s	remaining: 4m 10s
3201:	learn: 0.0533774	test: 0.0633197	best: 0.0633195 (3200)	total: 2m 40s	remaining: 4m 10s
3202:	learn: 0.0533756	test: 0.0633197	best: 0.0633195 (3200)	total: 2m 40s	remaining: 4m 9s
3203:	learn: 0.0533733	test: 0.0633195	best: 0.0633195 (3200)	total: 2m 40s	remaining: 4m 9s
3204:	learn: 0.0533692	test: 0.0633173	best: 0.0633173 (3204)	total: 2m 40s	remaining: 4m 9s
3205:	learn: 0.0533665	test: 0.0633172	best: 0.0633172 (3205)	total: 2m 40s	remaining: 4m 9s
3206:	learn: 0.0533646	test: 0.0633167	best: 0.0633167 (3206)	total: 2m 40s	remaining: 4m 9s
3207:	learn: 0

In [ ]:
meta_params = {'timestamp': dt.now().strftime('%Y-%m-%d %H:%M:%S'),
    'embeddings': model_name, 'db_name': db_name}
meta_params.update(params)
meta_params['best_iteration'] = model.best_iteration_

meta_params['train_loss'] = model.best_score_['learn'][params['loss_function']]
meta_params['val_loss'] = model.best_score_['validation'][params['loss_function']]
meta_params['test_loss'] = model.eval_metrics(test_pool, ['RMSE'])['RMSE'][-1]

meta_params['train_r2'] = model.score(train_embeddings, y_train)
meta_params['val_r2'] = model.score(val_embeddings, y_val)
meta_params['test_r2'] = model.score(test_embeddings, y_test)

end = time.time()
meta_params['time'] = end - start

run_results_file = 'run_results.csv'
run_results_df = (pd.read_csv('run_results.csv')
                  if os.path.exists(run_results_file)
                  else pd.DataFrame(columns=meta_params.keys()))
run_results_df.loc[len(run_results_df)] = meta_params
run_results_df.to_csv(run_results_file, index=False)
run_results_df = pd.read_csv(run_results_file)

# print 3 digits
pd.options.display.float_format = '{:.5f}'.format

print('Past Results')
print(run_results_df.sort_values(by='test_loss').head(16))
print()
print('This Run')
print(run_results_df.tail(1))

Past Results
              timestamp            embeddings             db_name  iterations  depth  learning_rate loss_function   l2_leaf_reg  best_iteration  train_loss  val_loss  test_loss  train_r2   val_r2  test_r2      time
10  2024-08-21 06:31:35  all-roberta-large-v1  need_for_speed_30D       16384      8        0.10000          RMSE    1000.00000           16244     0.04754   0.06613    0.06171   0.47546  0.07570  0.06185 765.58778
8   2024-08-21 06:10:51  all-roberta-large-v1  need_for_speed_30D        8192      6        0.10000          RMSE    1000.00000            8117     0.05620   0.06647    0.06182   0.26868  0.06614  0.05866 209.13300
9   2024-08-21 06:18:22  all-roberta-large-v1  need_for_speed_30D        8192      8        0.10000          RMSE    1000.00000            8168     0.05407   0.06629    0.06183   0.32379  0.07138  0.05825 415.64894
13  2024-08-21 06:53:00  all-roberta-large-v1  need_for_speed_30D        8192     10        0.25000          RMSE    1000.00000

In [ ]:
# dont use scientific notation
pd.options.display.float_format = '{:.3f}'.format
# model = model.load_model(f'/data/models/{model_name}_{db_name}_model.cbm')
temp_my_df = my_df.copy()
temp_my_df['predicted_views'] = model.predict(my_pool)
print(model.score(my_pool))
temp_my_df = temp_my_df[['video_title', 'video_view_count', 'predicted_views']]
temp_my_df.sort_values(by='predicted_views', ascending=False)

-0.2572151879446589


,video_title,video_view_count,predicted_views
884329,Need for Speed: Carbon - Retrospective,1.000,0.090
884330,Need for Speed: Most Wanted - Retrospective,0.514,0.066
884325,Need for Speed: Undercover - Retrospective,0.390,0.043
884332,Neon White is Stupid Good,0.006,0.030
884320,"Star Wars' New ""Controversy""",0.000,0.029
884324,Need for Speed: Underground 2 - Retrospective,0.216,0.029
884333,The Only Honest Game About War; This War of Mine,0.007,0.027
884327,Fallout: New Vegas Review (2024),0.013,0.023
884326,Need for Speed: ProStreet - Retrospective,0.575,0.022
884323,Need for Speed: The Run - Retrospective,0.231,0.022


In [ ]:
# save model
model.save_model(f'models/{model_name}_{db_name}_model_31\.cbm')


In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.embeddings.create(
    input="Your text string goes here",
    model="text-embedding-3-small"
)

print(response.data[0].embedding)

# Captioned MiniLM (Custom)

* MinMax Scale by channel

* Use full dataset

* Create custom embeddings from MiniLM

In [ ]:
# Input variables

load = True
captioned_only = True
sample = False

In [ ]:
# Preprocess data

%%time

from tqdm.notebook import tqdm
tqdm.pandas

from sklearn.preprocessing import MinMaxScaler

group_data = []
for name, group in tqdm(raw_df.groupby('channel_id')):
  if len(group) > 1:
    scaler = MinMaxScaler()
    row_data = group['video_view_count'].values.reshape(-1, 1)
    group['video_view_count'] = scaler.fit_transform(row_data)
    group_data.append(group)

  0%|          | 0/1652 [00:00<?, ?it/s]

CPU times: user 4.16 s, sys: 235 ms, total: 4.4 s
Wall time: 8.58 s


In [ ]:
# Split data

%%time

df = pd.concat(group_data).dropna(subset=['video_view_count'])

my_channel_title = 'Mitchell Diedrich'
my_df = df[df['channel_title'] == my_channel_title]
df = df[df['channel_title'] != my_channel_title]

if captioned_only:
  df = df[df['video_caption']==True]
if sample:
  df = df.sample(frac=0.1)

from sklearn.model_selection import train_test_split

train_df, val_test_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, random_state=42)

train_df.shape, val_df.shape, test_df.shape, my_df.shape

CPU times: user 2.64 s, sys: 555 ms, total: 3.2 s
Wall time: 3.22 s


((16459, 14), (2057, 14), (2058, 14), (15, 14))

In [ ]:
%%time

import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn

# Assuming you have your data as a DataFrame
titles = train_df['video_title'].to_numpy()
views = train_df['video_view_count'].to_numpy()

# Prepare data
data = pd.DataFrame({
    'title': titles,
    'views': views
})

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# modelname
model_name = 'all-MiniLM-L6-v2'

# Load SentenceTransformer model
base_model = SentenceTransformer(model_name)

save_path = f'models/embeddings/captioned_minilm.pt'

# Define a regression model
class RegressionModel(nn.Module):
    def __init__(self, transformer_model):
        super(RegressionModel, self).__init__()
        self.transformer = transformer_model
        try:
          self.regressor = nn.Linear(384, 1)  # Change embedding size to 384
        except:
          self.regressor = nn.Linear(1024, 1)  # Change embedding size to 1024

    def forward(self, texts):
        # SentenceTransformer requires tokenized inputs
        embeddings = self.transformer.encode(texts, convert_to_tensor=True, show_progress_bar=True).to(device)
        # Pass the embeddings through the regression layer to predict views
        output = self.regressor(embeddings)
        return output

# Instantiate the regression model
model = RegressionModel(base_model).to(device)

if load:
  model.load_state_dict(torch.load(save_path))
else:
  # Define loss function and optimizer
  criterion = nn.MSELoss().to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

  # Convert titles to embeddings and views to tensors
  train_titles = [row['title'] for index, row in data.iterrows()]
  train_views = torch.tensor(data['views'].values, dtype=torch.float32).unsqueeze(1).to(device)  # Reshape to (batch_size, 1)

  # Train the model
  model.train()
  for epoch in range(1):
      optimizer.zero_grad()

      # Forward pass
      predictions = model(train_titles)

      # Compute loss
      loss = criterion(predictions, train_views)

      # Backward pass and optimization
      loss.backward()
      optimizer.step()

      print(f"Epoch {epoch+1}, Loss: {loss.item()}")

  # save model as embedding model

  torch.save(model.state_dict(), save_path)

  print("Training complete.")

CPU times: user 233 ms, sys: 163 ms, total: 396 ms
Wall time: 6.08 s


In [ ]:
%%time

import numpy as np
from sentence_transformers import SentenceTransformer

embeddings_name = '_embeddings_captioned_minilm_custom.npy'

if load:

  train_embeddings = np.load(f'models/embeddings/train{embeddings_name}',
                           allow_pickle=True)
  val_embeddings = np.load(f'models/embeddings/val{embeddings_name}',
                          allow_pickle=True)
  test_embeddings = np.load(f'models/embeddings/test{embeddings_name}',
                            allow_pickle=True)
  my_embeddings = np.load(f'models/embeddings/my{embeddings_name}',
                          allow_pickle=True)
else:
  size = 64
  # Extract embeddings (using only the transformer part of your model)
  with torch.no_grad():
    train_embeddings = model.transformer.encode(
        titles,
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=size
        ).to(device)
    val_embeddings = model.transformer.encode(
        val_df['video_title'].to_numpy(),
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=size
        ).to(device)
    test_embeddings = model.transformer.encode(
        test_df['video_title'].to_numpy(),
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=size
        ).to(device)
    my_embeddings = model.transformer.encode(
        my_df['video_title'].to_numpy(),
        convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=size
        ).to(device)

  # convert train_embeddings to np array
  train_embeddings = train_embeddings.cpu().numpy()
  val_embeddings = val_embeddings.cpu().numpy()
  test_embeddings = test_embeddings.cpu().numpy()
  my_embeddings = my_embeddings.cpu().numpy()

  import numpy as np
  # write embeddings to files
  np.save(f'models/embeddings/train{embeddings_name}', train_embeddings)
  np.save(f'models/embeddings/val{embeddings_name}', val_embeddings)
  np.save(f'models/embeddings/test{embeddings_name}', test_embeddings)
  np.save(f'models/embeddings/my{embeddings_name}', my_embeddings)

train_embeddings.shape, val_embeddings.shape, test_embeddings.shape, my_embeddings.shape

CPU times: user 18.2 ms, sys: 30 ms, total: 48.2 ms
Wall time: 3.03 s


((16459, 384), (2057, 384), (2058, 384), (15, 384))

In [ ]:
# Fit model

%%time

pd.set_option('display.width', 1000)

from datetime import datetime as dt
from catboost import CatBoostRegressor, Pool
import time

y_train = train_df['video_view_count'].to_numpy()
y_val = val_df['video_view_count'].to_numpy()
y_test = test_df['video_view_count'].to_numpy()
y_my = my_df['video_view_count'].to_numpy()

train_pool = Pool(train_embeddings, y_train)
val_pool = Pool(val_embeddings, y_val)
test_pool = Pool(test_embeddings, y_test)
my_pool = Pool(my_embeddings, y_my)

start = time.time()

params = {
    'iterations': 2048,
    'depth': 8,
    'learning_rate': .1,
    'loss_function': 'RMSE',
    'l2_leaf_reg': 100,
}

model = CatBoostRegressor(**params, random_seed=42, verbose=True, task_type='GPU')
model.fit(train_pool, eval_set=val_pool, verbose=True)

loss_data = {
    'train': model.eval_metrics(train_pool, ['RMSE'])['RMSE'][-1],
    'val': model.eval_metrics(val_pool, ['RMSE'])['RMSE'][-1],
    'test': model.eval_metrics(test_pool, ['RMSE'])['RMSE'][-1]
}
r2_data = {
    'train': model.score(train_embeddings, y_train),
    'val': model.score(val_embeddings, y_val),
    'test': model.score(test_embeddings, y_test)
}

loss_series = pd.Series(loss_data, name='loss')
r2_series = pd.Series(r2_data, name='r2')

eta_params = {'timestamp': dt.now().strftime('%Y-%m-%d %H:%M:%S'),
    'embeddings': model_name, 'db_name': db_name}
meta_params.update(params)
meta_params['best_iteration'] = model.best_iteration_

meta_params['train_loss'] = model.best_score_['learn'][params['loss_function']]
meta_params['val_loss'] = model.best_score_['validation'][params['loss_function']]
meta_params['test_loss'] = model.eval_metrics(test_pool, ['RMSE'])['RMSE'][-1]

meta_params['train_r2'] = model.score(train_embeddings, y_train)
meta_params['val_r2'] = model.score(val_embeddings, y_val)
meta_params['test_r2'] = model.score(test_embeddings, y_test)

end = time.time()
meta_params['time'] = end - start

run_results_file = 'captioned_minilm_run_results.csv'
run_results_df = (pd.read_csv(run_results_file)
                  if os.path.exists(run_results_file)
                  else pd.DataFrame(columns=meta_params.keys()))
run_results_df.loc[len(run_results_df)] = meta_params
run_results_df.to_csv(run_results_file, index=False)
run_results_df = pd.read_csv(run_results_file)

# print 3 digits
pd.options.display.float_format = '{:.5f}'.format

print('Past Results')
print(run_results_df.sort_values(by='test_loss').head(16))
print()

metrics_df = pd.concat([loss_series, r2_series], axis=1)
metrics_df.T

0:	learn: 0.0807856	test: 0.0930953	best: 0.0930953 (0)	total: 189ms	remaining: 6m 27s
1:	learn: 0.0802579	test: 0.0926788	best: 0.0926788 (1)	total: 330ms	remaining: 5m 37s
2:	learn: 0.0797277	test: 0.0922143	best: 0.0922143 (2)	total: 436ms	remaining: 4m 57s
3:	learn: 0.0793755	test: 0.0919102	best: 0.0919102 (3)	total: 530ms	remaining: 4m 31s
4:	learn: 0.0789341	test: 0.0915945	best: 0.0915945 (4)	total: 647ms	remaining: 4m 24s
5:	learn: 0.0785477	test: 0.0913046	best: 0.0913046 (5)	total: 760ms	remaining: 4m 18s
6:	learn: 0.0782199	test: 0.0910302	best: 0.0910302 (6)	total: 862ms	remaining: 4m 11s
7:	learn: 0.0778818	test: 0.0907804	best: 0.0907804 (7)	total: 965ms	remaining: 4m 6s
8:	learn: 0.0776941	test: 0.0905993	best: 0.0905993 (8)	total: 1.05s	remaining: 3m 58s
9:	learn: 0.0774304	test: 0.0904232	best: 0.0904232 (9)	total: 1.16s	remaining: 3m 55s
10:	learn: 0.0771025	test: 0.0902419	best: 0.0902419 (10)	total: 1.27s	remaining: 3m 56s
11:	learn: 0.0768017	test: 0.0900335	best:

,train,val,test
loss,0.03487,0.08084,0.07215
r2,0.81625,0.25463,0.21736


In [ ]:
meta_params = {'timestamp': dt.now().strftime('%Y-%m-%d %H:%M:%S'),
    'embeddings': model_name, 'db_name': db_name}
meta_params.update(params)
meta_params['best_iteration'] = model.best_iteration_

meta_params['train_loss'] = model.best_score_['learn'][params['loss_function']]
meta_params['val_loss'] = model.best_score_['validation'][params['loss_function']]
meta_params['test_loss'] = model.eval_metrics(test_pool, ['RMSE'])['RMSE'][-1]

meta_params['train_r2'] = model.score(train_embeddings, y_train)
meta_params['val_r2'] = model.score(val_embeddings, y_val)
meta_params['test_r2'] = model.score(test_embeddings, y_test)

end = time.time()
meta_params['time'] = end - start

run_results_file = 'run_results.csv'
run_results_df = (pd.read_csv('run_results.csv')
                  if os.path.exists(run_results_file)
                  else pd.DataFrame(columns=meta_params.keys()))
run_results_df.loc[len(run_results_df)] = meta_params
run_results_df.to_csv(run_results_file, index=False)
run_results_df = pd.read_csv(run_results_file)

# print 3 digits
pd.options.display.float_format = '{:.5f}'.format

print('Past Results')
print(run_results_df.sort_values(by='test_loss').head(16))
print()
print('This Run')
print(run_results_df.tail(1))

In [ ]:
rm iter_captioned_minilm_run_results.csv

In [ ]:
# Fit model

%%time

import pandas as pd

run_results_file = 'iter_captioned_minilm_run_results.csv'

pd.set_option('display.width', 1000)

from datetime import datetime as dt
from catboost import CatBoostRegressor, Pool
import time

from tqdm.notebook import tqdm
tqdm.pandas()

from scipy import stats

y_train = train_df['video_view_count'].to_numpy()
y_val = val_df['video_view_count'].to_numpy()
y_test = test_df['video_view_count'].to_numpy()
y_my = my_df['video_view_count'].to_numpy()

train_pool = Pool(train_embeddings, y_train)
val_pool = Pool(val_embeddings, y_val)
test_pool = Pool(test_embeddings, y_test)
my_pool = Pool(my_embeddings, y_my)

iterations = [1024]

pbar = tqdm(total=sum(iterations))

for iteration in iterations:

  start = time.time()


  params = {
      'iterations': iteration,
      'depth': 8,
      'learning_rate': .05,
      'l2_leaf_reg': 100,
      'random_strength': 10,
  }

  print(params)

  model = CatBoostRegressor(**params,
                            random_seed=42,
                            verbose=True,
                            task_type='GPU',
                            loss_function='RMSE',
                            eval_metric='R2',
                            metric_period=64,
                            )
  model.fit(train_pool, eval_set=val_pool, verbose=True)

  loss_data = {
      'train': model.eval_metrics(train_pool, ['RMSE'])['RMSE'][-1],
      'val': model.eval_metrics(val_pool, ['RMSE'])['RMSE'][-1],
      'test': model.eval_metrics(test_pool, ['RMSE'])['RMSE'][-1]
  }
  r2_data = {
      'train': model.score(train_embeddings, y_train),
      'val': model.score(val_embeddings, y_val),
      'test': model.score(test_embeddings, y_test)
  }
  p_data = {
      'train': 0,
      'val': val_p,
      'test': test_p
  }

  loss_series = pd.Series(loss_data, name='loss')
  r2_series = pd.Series(r2_data, name='r2')
  p_series = pd.Series(p_data, name='p')
  metrics_df = pd.concat([loss_series, r2_series, p_series], axis=1)
  print(metrics_df.T)
  print()

  y_pred_train = model.predict(train_pool)
  y_pred_val = model.predict(val_pool)
  y_pred_test = model.predict(test_pool)

  train_residuals = y_train - y_pred_train
  val_residuals = y_val - y_pred_val
  test_residuals = y_test - y_pred_test

  val_t, val_p = stats.mannwhitneyu(train_residuals, val_residuals)
  test_t, test_p = stats.mannwhitneyu(train_residuals, test_residuals)

  params['best_iteration'] = model.best_iteration_

  params['train_loss'] = model.best_score_['learn']['RMSE']
  params['val_loss'] = model.best_score_['validation']['RMSE']
  params['val_p'] = val_p

  params['train_r2'] = model.score(train_embeddings, y_train)
  params['val_r2'] = model.score(val_embeddings, y_val)
  params['test_r2'] = model.score(test_embeddings, y_test)

  params['time'] = time.time() - start

  run_results_df = (pd.read_csv(run_results_file)
                    if os.path.exists(run_results_file)
                    else pd.DataFrame(columns=params.keys()))

  run_results_df.loc[len(run_results_df)] = params
  run_results_df.to_csv(run_results_file, index=False)

  pbar.update(iteration)

pbar.close()

run_results_df = pd.read_csv(run_results_file)

# print 3 digits
pd.options.display.float_format = '{:.5f}'.format

remap = {
    'iterations': 'iter',
    'learning_rate': 'lr',
    'loss_function': 'loss',
    'l2_leaf_reg': 'l2',
    'best_iteration': 'top_iter',
}

run_results_df.rename(columns=remap, inplace=True)

pd.set_option('display.max_rows', None)

print('Past Results')
print(run_results_df)
print()

  0%|          | 0/1024 [00:00<?, ?it/s]

{'iterations': 1024, 'depth': 8, 'learning_rate': 0.05, 'l2_leaf_reg': 100, 'random_strength': 10}


Metric R2 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.0055661	test: 0.0029995	best: 0.0029995 (0)	total: 197ms	remaining: 3m 21s
64:	learn: 0.1746802	test: 0.1243122	best: 0.1243122 (64)	total: 6.61s	remaining: 1m 37s
128:	learn: 0.2294795	test: 0.1531207	best: 0.1531207 (128)	total: 14.4s	remaining: 1m 39s
192:	learn: 0.2647828	test: 0.1676837	best: 0.1676837 (192)	total: 20.8s	remaining: 1m 29s
256:	learn: 0.2987036	test: 0.1776739	best: 0.1776739 (256)	total: 28s	remaining: 1m 23s
320:	learn: 0.3300412	test: 0.1850144	best: 0.1850144 (320)	total: 34.4s	remaining: 1m 15s
384:	learn: 0.3544407	test: 0.1922712	best: 0.1922712 (384)	total: 41.1s	remaining: 1m 8s
448:	learn: 0.3793719	test: 0.1978704	best: 0.1978704 (448)	total: 47.3s	remaining: 1m
512:	learn: 0.3987404	test: 0.2015775	best: 0.2015775 (512)	total: 53.1s	remaining: 52.9s
576:	learn: 0.4172180	test: 0.2044605	best: 0.2044605 (576)	total: 1m	remaining: 46.5s
640:	learn: 0.4334769	test: 0.2082419	best: 0.2082419 (640)	total: 1m 5s	remaining: 39.3s
704:	learn: 0.4494

In [ ]:
      # dont use scientific notation
pd.options.display.float_format = '{:.3f}'.format
# find model with '33' in name
file_names = os.listdir('/data/models')
file_names = [f for f in file_names if '33' in f]
file_name = file_names[0]
model = model.load_model(f'/data/models/{file_name}')

# model = model.load_model(f'/data/models/{model_name}_{db_name}_model.cbm')
temp_my_df = my_df.copy()
temp_my_df['predicted_views'] = model.predict(my_pool)
print(model.score(my_pool))
temp_my_df = temp_my_df[['video_title', 'video_view_count', 'predicted_views']]
temp_my_df.sort_values(by='predicted_views', ascending=False)

FileNotFoundError: [Errno 2] No such file or directory: '/data/models'

In [ ]:
# save model
model.save_model(f'models/{model_name}_{db_name}_model_31\.cbm')
